# Huấn luyện Hồi quy tuyến tính cho lương CNTT

Notebook nay minh hoa baseline `sklearn.linear_model.LinearRegression` cho du bao luong IT tu `data/analysis/salary_analysis_clean.csv`.

Notebook là luồng huấn luyện chính. Chạy lần lượt các cell để kiểm tra dữ liệu, huấn luyện mô hình, đánh giá kết quả và ghi artifact cho Streamlit.

## Caveats

- Day la mo hinh nen de hoc va demo, khong phai ket luan luong thi truong.
- Dataset salary hien chi gom job co numeric salary. TopDev hien khong co numeric salary nen khong vao model.
- Target dung `log_salary_monthly_vnd` de giam skew va outlier influence.
- Coefficient cua Linear Regression chi la tin hieu trong model, khong phai bang chung nhan qua.

## 0. Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "analysis" / "salary_analysis_clean.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/analysis/salary_analysis_clean.csv")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SALARY_PATH = REPO_ROOT / "data" / "analysis" / "salary_analysis_clean.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "modeling" / "salary_regression" / "safe_baseline"

print(f"Repo root: {REPO_ROOT}")
print(f"Salary input: {SALARY_PATH}")
print(f"Model output dir: {OUTPUT_DIR}")

Repo root: G:\project\Vietnam IT Job Market Intelligence
Salary input: G:\project\Vietnam IT Job Market Intelligence\data\analysis\salary_analysis_clean.csv
Model output dir: G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline


In [2]:
from modeling.salary_regression import (
    ARTIFACT_FILENAMES,
    MODEL_FILENAME,
    fit_salary_linear_regression,
    prepare_salary_modeling_data,
    write_outputs,
)

## 1. Load Data

Cell nay kiem tra nhanh dung file, so dong va coverage source/currency truoc khi train.

In [3]:
salary = pd.read_csv(SALARY_PATH, encoding="utf-8-sig")

print(f"Rows: {len(salary):,}")
print(f"Unique URLs: {salary['url'].nunique():,}")
assert len(salary) > 0, "salary_analysis_clean.csv must contain rows"
assert salary["url"].is_unique, "salary dataset should be deduped by URL"

display(salary[["source", "title", "company", "location", "salary_raw", "salary_currency", "salary_period", "salary_midpoint", "seniority", "experience_min"]].head(10))
display(pd.crosstab(salary["source"], salary["salary_currency"], dropna=False))

Rows: 504
Unique URLs: 504


,source,title,company,location,salary_raw,salary_currency,salary_period,salary_midpoint,seniority,experience_min
0,itviec,10 Fullstack Deverloper (Java/ Spring Boot/Angular),Thien Hoang Solutions JSC,Hà Nội,1200 - 1600 USD,USD,month,1400.0,middle,4.0
1,itviec,[10] Nhân sự đánh giá rò quét lỗ hổng ANTT (Pentest),TRUNG TÂM THÔNG TIN TÍN DỤNG QUỐC GIA VIỆT NAM (CIC),Hà Nội,800 - 1500 USD,USD,month,1150.0,middle,2.0
2,itviec,[11] Nhân sự săn tìm mối nguy (Thread huntting),TRUNG TÂM THÔNG TIN TÍN DỤNG QUỐC GIA VIỆT NAM (CIC),Hà Nội,800 - 1500 USD,USD,month,1150.0,middle,2.0
3,itviec,"20 Java Backend Devs (Spring Boot, Hibernate, Oracle)",Thien Hoang Solutions JSC,Hà Nội,1100 - 1900 USD,USD,month,1500.0,middle,3.0
4,itviec,AI Agent Engineer (Contract): Build the Agent Layer,Panto Martech LLC,Hồ Chí Minh,2500 - 4000 USD,USD,month,3250.0,NaN,NaN
5,itviec,"AI Agent Engineer (Python, LLM)",KDS Vietnam,Hồ Chí Minh,1300 - 1800 USD,USD,month,1550.0,middle,3.0
6,itviec,AI Engineer (Định hướng Data Scientist),KALAPA,Hà Nội,600 - 1300 USD,USD,month,950.0,middle,2.0
7,itviec,Middle AI Engineer (Inference Engineering),Autonomous,Hồ Chí Minh,1000 - 2000 USD,USD,month,1500.0,middle,1.0
8,itviec,AI Engineer (Machine Learning/LLM/API),CLUEGA,Hồ Chí Minh,"18M-65,5M",VND,month,41750000.0,middle,2.0
9,itviec,AI Engineer with Leading Korean AI Entrepreneurs,Abc Studio Việt Nam,Hồ Chí Minh,1000 - 2000 USD,USD,month,1500.0,middle,3.0


salary_currency,USD,VND
source,,
itviec,176,26
topcv,6,296


## 2. Prepare Modeling Data

Backend helper lam cac viec chinh: convert salary ve VND/thang, tao target log salary, normalize location/skills, tao feature flags, va chan leakage columns.

In [4]:
modeling_data = prepare_salary_modeling_data(
    salary,
    top_skills=30,
    min_skill_count=5,
    usd_to_vnd=26_000,
)

print(f"Modeling rows: {len(modeling_data.frame):,}")
print(f"Feature count: {len(modeling_data.feature_columns):,}")
print(f"Top skills: {modeling_data.top_skills[:10]}")
display(modeling_data.audit.head(25))
display(modeling_data.frame[["source", "salary_raw", "salary_currency", "salary_period", "salary_period_clean", "salary_monthly_vnd_m", "location_norm", "skill_count", "log_salary_monthly_vnd"]].head(10))

Modeling rows: 504
Feature count: 37
Top skills: ['sql', 'python', 'java', 'docker', 'aws', 'javascript', 'postgresql', 'react', 'devops', 'mysql']


,metric,value
0,input_rows,504.0000
1,modeling_rows,504.0000
2,dropped_invalid_salary_rows,0.0000
3,usd_to_vnd,26000.0000
4,salary_period_year_without_annual_signal,101.0000
5,salary_period_annual_signal_rows,0.0000
6,top_skill_count,30.0000
7,feature_count,37.0000
8,missing_rate_experience_min,0.1389
9,missing_rate_skill_count,0.0000


,source,salary_raw,salary_currency,salary_period,salary_period_clean,salary_monthly_vnd_m,location_norm,skill_count,log_salary_monthly_vnd
0,itviec,1200 - 1600 USD,USD,month,month,36.40,Ha Noi,12,17.410079
1,itviec,800 - 1500 USD,USD,month,month,29.90,Ha Noi,3,17.213369
2,itviec,800 - 1500 USD,USD,month,month,29.90,Ha Noi,3,17.213369
3,itviec,1100 - 1900 USD,USD,month,month,39.00,Ha Noi,14,17.479072
4,itviec,2500 - 4000 USD,USD,month,month,84.50,Ho Chi Minh,7,18.252262
5,itviec,1300 - 1800 USD,USD,month,month,40.30,Ho Chi Minh,7,17.511862
6,itviec,600 - 1300 USD,USD,month,month,24.70,Ha Noi,4,17.022314
7,itviec,1000 - 2000 USD,USD,month,month,39.00,Ho Chi Minh,7,17.479072
8,itviec,"18M-65,5M",VND,month,month,41.75,Ho Chi Minh,10,17.547210
9,itviec,1000 - 2000 USD,USD,month,month,39.00,Ho Chi Minh,6,17.479072


## 3. Train Linear Regression

In [5]:
result = fit_salary_linear_regression(
    modeling_data,
    test_size=0.2,
    random_state=42,
)

print(f"Train rows: {result.train_rows:,}")
print(f"Test rows: {result.test_rows:,}")
display(result.metrics)

Train rows: 403
Test rows: 101


c:\Users\VNAT\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,scope,group_column,group_value,n,mae_log,rmse_log,r2_log,mae_million_vnd,rmse_million_vnd,median_abs_error_million_vnd
0,overall,,,101,0.365949,0.476153,0.525292,10.432397,15.149481,7.691514
1,group,source,itviec,41,0.301217,0.363109,0.595987,13.301106,18.819736,9.457001
2,group,source,topcv,60,0.410183,0.539955,0.083402,8.472112,12.012981,5.884123
3,group,seniority,intern,4,0.265356,0.311978,-0.214243,2.130878,2.768272,1.557871
4,group,seniority,junior,21,0.380836,0.438136,0.019869,7.331403,9.156957,6.035946
5,group,seniority,lead,13,0.277615,0.312374,0.802857,16.125257,25.784908,7.819501
6,group,seniority,middle,34,0.357109,0.460691,-0.271903,9.759238,12.943489,7.132966
7,group,seniority,senior,16,0.263838,0.330118,0.393178,12.048296,15.147668,9.748869
8,group,seniority,Unknown,13,0.609985,0.796633,-0.210941,12.074918,16.119985,11.046198


## 4. Inspect Prediction Errors

`abs_error_million_vnd` cho biet job nao model du doan lech nhieu nhat trong test set.

In [6]:
display(result.predictions.head(20))
display(result.predictions["abs_error_million_vnd"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame())

,url,title,company,source,location,location_norm,seniority,experience_min,actual_log_salary,predicted_log_salary,actual_salary_million_vnd,predicted_salary_million_vnd,residual_million_vnd,abs_error_million_vnd
0,https://itviec.com/it-jobs/principal-backend-engineer-aws-sql-rowboat-software-0838,"Principal Backend Engineer (AWS, TypeScript, DynamoDB)",ROWBOAT SOFTWARE,itviec,Hồ Chí Minh,Ho Chi Minh,lead,10.0,19.019517,18.445668,182.0,102.530158,79.469842,79.469842
1,https://www.topcv.vn/viec-lam/ktv-lap-trinh-cnc/2203511.html,KTV Lập Trình CNC,CÔNG TY TNHH CÔNG NGHỆ KỸ THUẬT CAO HI-P VIỆT NAM,topcv,Bắc Ninh,Bắc Ninh,middle,3.0,16.454568,17.848556,14.0,56.432489,-42.432489,42.432489
2,https://www.topcv.vn/viec-lam/truong-phong-it/2217809.html,Trưởng Phòng IT,CÔNG TY CỔ PHẦN PHÁT TRIỂN CƠ ĐIỆN LẠNH HOÀNG ĐẠT,topcv,Ha Noi,Ha Noi,<NA>,NaN,17.727534,16.553452,50.0,15.455132,34.544868,34.544868
3,https://www.topcv.vn/viec-lam/nhan-vien-tu-van-cham-soc-khach-hang/2214791.html,Nhân Viên Tư Vấn Chăm Sóc Khách Hàng,CÔNG TY TNHH CÔNG NGHỆ PHẦN MỀM NASANI,topcv,Cần Thơ,Cần Thơ,<NA>,NaN,16.213406,17.591345,11.0,43.633912,-32.633912,32.633912
4,https://itviec.com/it-jobs/hn-senior-expert-cloud-infrastructure-engineer-one-mount-group-1447,(HN) Senior/Expert Cloud Infrastructure Engineer,One Mount Group,itviec,Hà Nội,Ha Noi,senior,5.0,18.172219,17.680827,78.0,47.718383,30.281617,30.281617
5,https://itviec.com/it-jobs/product-owner-sales-intelligence-platform-agile-fpt-digital-1328,Product Owner - Sales Intelligence Platform (Agile),FPT Digital,itviec,Hà Nội,Ha Noi,senior,5.0,18.085208,17.562014,71.5,42.372650,29.127350,29.127350
6,https://itviec.com/it-jobs/information-security-governance-manager-cong-ty-co-phan-chung-khoan-kis-viet-nam-2215,Information Security Governance Manager,Công ty Cổ phần Chứng khoán KIS Việt Nam,itviec,Hồ Chí Minh,Ho Chi Minh,lead,NaN,17.989898,17.434635,65.0,37.304887,27.695113,27.695113
7,https://itviec.com/it-jobs/data-tech-lead-sql-python-data-modeling-aws-cong-ty-tnhh-stravo-vietnam-4341,"Data Tech Lead (SQL, Python, Data Modeling, AWS)",Công ty TNHH Stravo Vietnam,itviec,Hồ Chí Minh,Ho Chi Minh,lead,7.0,18.252262,18.531441,84.5,111.712673,-27.212673,27.212673
8,https://itviec.com/it-jobs/senior-product-designer-ux-ui-ai-huenx-vietnam-1009,Senior Product Designer (UX/UI + AI),HUENX VIETNAM,itviec,Hồ Chí Minh,Ho Chi Minh,senior,7.0,17.766754,18.177171,52.0,78.387199,-26.387199,26.387199
9,https://www.topcv.vn/viec-lam/chuyen-vien-kinh-doanh-sale-telesale-co-kinh-nghiem-linh-vuc-it-phan-mem-luong-cung-up...,Chuyên Viên Kinh Doanh/ Sale /Telesale (Có Kinh Nghiệm Lĩnh Vực IT - Phần Mềm ) - Lương Cứng Up To 25 Triệu - Tại Hà...,CÔNG TY CỔ PHẦN CÔNG NGHỆ SALELY AI,topcv,Ha Noi,Ha Noi,middle,2.0,17.622173,16.791432,45.0,19.607674,25.392326,25.392326


,abs_error_million_vnd
count,101.000000
mean,10.432397
std,11.039863
min,0.157692
25%,3.049995
50%,7.691514
75%,13.502900
90%,23.497089
95%,29.127350
max,79.469842


## 5. Inspect Coefficients

Doc coefficient nhu tin hieu baseline. Feature one-hot va feature correlated khong nen duoc doc nhu quan he nhan qua.

In [7]:
display(result.coefficients.head(30))
display(result.coefficients.sort_values("coefficient", ascending=False).head(15))
display(result.coefficients.sort_values("coefficient", ascending=True).head(15))

,feature,coefficient,abs_coefficient
0,categorical__location_norm_Gia Lai,-1.753238,1.753238
1,categorical__location_norm_Tây Ninh,-1.373814,1.373814
2,categorical__location_norm_Khánh Hòa,-1.277111,1.277111
3,categorical__location_norm_Thành phố khác,-1.146221,1.146221
4,categorical__seniority_intern,-1.094569,1.094569
5,categorical__location_norm_Hải Phòng,-1.073469,1.073469
6,categorical__location_norm_Da Nang,-1.066476,1.066476
7,categorical__location_norm_Ho Chi Minh,-0.970000,0.970000
8,categorical__location_norm_Ha Noi,-0.958594,0.958594
9,categorical__location_norm_Hưng Yên,-0.958546,0.958546


,feature,coefficient,abs_coefficient
13,categorical__location_norm_Nhật Bản,0.419406,0.419406
14,skill__skill__php,0.390631,0.390631
15,categorical__seniority_lead,0.338469,0.338469
16,categorical__seniority_senior,0.325309,0.325309
19,skill__skill__giao_tiep,0.233606,0.233606
20,skill__skill__c_sharp,0.229632,0.229632
21,skill__skill__english,0.224353,0.224353
23,categorical__work_mode_remote,0.185316,0.185316
24,categorical__location_norm_Nước Ngoài,0.185105,0.185105
25,categorical__seniority_middle,0.184116,0.184116


,feature,coefficient,abs_coefficient
0,categorical__location_norm_Gia Lai,-1.753238,1.753238
1,categorical__location_norm_Tây Ninh,-1.373814,1.373814
2,categorical__location_norm_Khánh Hòa,-1.277111,1.277111
3,categorical__location_norm_Thành phố khác,-1.146221,1.146221
4,categorical__seniority_intern,-1.094569,1.094569
5,categorical__location_norm_Hải Phòng,-1.073469,1.073469
6,categorical__location_norm_Da Nang,-1.066476,1.066476
7,categorical__location_norm_Ho Chi Minh,-0.970000,0.970000
8,categorical__location_norm_Ha Noi,-0.958594,0.958594
9,categorical__location_norm_Hưng Yên,-0.958546,0.958546


## 6. Lưu model đã huấn luyện cho Streamlit

Sau cell này, Streamlit có thể nạp `model.joblib` trong `data/modeling/salary_regression/safe_baseline`.

In [8]:
write_outputs(result, OUTPUT_DIR)

artifact_rows = []
for filename in ARTIFACT_FILENAMES:
    path = OUTPUT_DIR / filename
    artifact_rows.append(
        {
            "artifact": filename,
            "exists": path.exists(),
            "path": path,
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        }
    )

print(f"Notebook trained model file: {OUTPUT_DIR / MODEL_FILENAME}")
display(pd.DataFrame(artifact_rows))

Notebook trained model file: G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline\model.joblib


,artifact,exists,path,size_kb
0,model.joblib,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline\model.joblib,16.5
1,metrics.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline\metrics.csv,1.3
2,predictions_test.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline\predictions_test.csv,32.4
3,coefficients.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline\coefficients.csv,3.7
4,data_audit.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\safe_baseline\data_audit.csv,1.2


## 7. Cách sử dụng

Mở notebook bằng môi trường dự án:

```bash
.\.venv\Scripts\jupyter.exe notebook notebooks/04_salary_linear_regression_training.ipynb
```

Chạy Streamlit sau khi đã chạy tất cả cell train:

```bash
.\.venv\Scripts\python.exe -m streamlit run streamlit_salary_regression_opencode.py
```